Install and Imports

In [18]:
!pip install -U transformers==4.57.3 accelerate datasets seqeval --quiet

import os, re, json, ast
from pathlib import Path
import numpy as np
import pandas as pd
import torch

os.environ["WANDB_DISABLED"] = "true"   # hard kill W&B

from datasets import Dataset, DatasetDict, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from seqeval.metrics import precision_score, recall_score, f1_score

from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/CS685/linkedin")
MODEL_NAME = "distilbert-base-uncased"

train_path = BASE / "ner_data_v3/bio_train.csv"
dev_path   = BASE / "ner_data_v3/bio_dev.csv"
test_path  = BASE / "ner_data_v3/bio_test.csv"

print("Transformers:", __import__("transformers").__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Transformers: 4.57.3


Load BIO csvs

In [19]:
def parse_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x) or str(x).strip() == "":
        return []
    return ast.literal_eval(x)

train_df = pd.read_csv(train_path)
dev_df   = pd.read_csv(dev_path)
test_df  = pd.read_csv(test_path)

for df_ in [train_df, dev_df, test_df]:
    df_["tokens"] = df_["tokens"].apply(parse_list)
    df_["tags"]   = df_["tags"].apply(parse_list)

print("Train/Dev/Test:", len(train_df), len(dev_df), len(test_df))
print(train_df.head(2))

Train/Dev/Test: 490 105 105
       job_id        sent_id domain  \
0  3895240684  3895240684_22    SWE   
1  3904367120   3904367120_1    SWE   

                                            sentence  \
0  5+ years experience with Cloud technology: GCP...   
1  Required Qualifications: Gitlab Datadog – APM,...   

                                              tokens  \
0  [5, +, years, experience, with, Cloud, technol...   
1  [Required, Qualifications, :, Gitlab, Datadog,...   

                                                tags  \
0  [O, O, O, O, O, B-SKILL, I-SKILL, O, B-SKILL, ...   
1  [O, O, O, O, O, O, B-SKILL, O, B-SKILL, I-SKIL...   

                                    gold_spans  \
0  ['Cloud technology', 'GCP', 'AWS', 'Azure']   
1     ['Github Datadog', 'APM', 'Log Tracing']   

                                  matched_char_spans  
0  [(25, 41, 'Cloud technology'), (43, 46, 'GCP')...  
1         [(42, 45, 'APM'), (47, 58, 'Log Tracing')]  


In [20]:
#labels
all_tags = set()
for df_ in [train_df, dev_df, test_df]:
    for tags in df_["tags"]:
        all_tags.update(tags)

label_list = sorted(list(all_tags))
print("Labels:", label_list)

label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

Labels: ['B-SKILL', 'I-SKILL', 'O']


In [21]:
# HF datasets
train_ds = Dataset.from_pandas(train_df[["tokens","tags"]])
dev_ds   = Dataset.from_pandas(dev_df[["tokens","tags"]])
test_ds  = Dataset.from_pandas(test_df[["tokens","tags"]])

raw = DatasetDict({"train": train_ds, "validation": dev_ds, "test": test_ds})
raw

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 490
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 105
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 105
    })
})

Tokenization + Label Alignment

In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding=False
    )

    aligned_labels = []
    for i, word_labels in enumerate(examples["tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(label2id[word_labels[word_id]])
            else:
                # subword continuation -> ignore in loss
                label_ids.append(-100)
            prev_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

tokenized = raw.map(tokenize_and_align_labels, batched=True)
tokenized

Map:   0%|          | 0/490 [00:00<?, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 490
    })
    validation: Dataset({
        features: ['tokens', 'tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 105
    })
    test: Dataset({
        features: ['tokens', 'tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 105
    })
})

Model + Collator

In [23]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Strict + Lenient Metrics

In [24]:
def tags_to_spans(tag_seq):
    # BIO spans as (type, start, end) where start/end are indices in word-level tag sequence.
    spans = []
    i = 0
    n = len(tag_seq)
    while i < n:
        t = tag_seq[i]
        if t.startswith("B-"):
            typ = t[2:]
            start = i
            j = i + 1
            while j < n and tag_seq[j] == f"I-{typ}":
                j += 1
            end = j - 1
            spans.append((typ, start, end))
            i = j
        else:
            i += 1
    return spans

def lenient_prf(pred_seqs, gold_seqs):
    # Lenient = any overlap between predicted and gold span of same type counts as match. 1-to-1 matching.

    tp = fp = fn = 0
    for pred_tags, gold_tags in zip(pred_seqs, gold_seqs):
        ps = tags_to_spans(pred_tags)
        gs = tags_to_spans(gold_tags)

        matched_gold = set()
        for (pl, ps_, pe_) in ps:
            hit = False
            for gi, (gl, gs_, ge_) in enumerate(gs):
                if gi in matched_gold:
                    continue
                if pl != gl:
                    continue
                if not (pe_ < gs_ or ps_ > ge_):  # overlap
                    hit = True
                    matched_gold.add(gi)
                    break
            if hit: tp += 1
            else:   fp += 1

        fn += (len(gs) - len(matched_gold))

    P = tp/(tp+fp) if (tp+fp) else 0.0
    R = tp/(tp+fn) if (tp+fn) else 0.0
    F1 = (2*P*R/(P+R)) if (P+R) else 0.0
    return P, R, F1

def align_predictions(logits, label_ids):
    # Convert token-level logits + token labels (-100 masked) -> word-level tag sequences for seqeval.

    pred_ids = np.argmax(logits, axis=-1)
    pred_tags, gold_tags = [], []

    for p_seq, l_seq in zip(pred_ids, label_ids):
        p_out, g_out = [], []
        for p, l in zip(p_seq, l_seq):
            if l == -100:
                continue
            p_out.append(label_list[p])
            g_out.append(label_list[l])
        pred_tags.append(p_out)
        gold_tags.append(g_out)

    return pred_tags, gold_tags

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred_tags, gold_tags = align_predictions(logits, labels)

    # Strict: seqeval entity-level (exact match spans)
    sP = precision_score(gold_tags, pred_tags)
    sR = recall_score(gold_tags, pred_tags)
    sF1 = f1_score(gold_tags, pred_tags)

    # Lenient: overlap match
    lP, lR, lF1 = lenient_prf(pred_tags, gold_tags)

    return {
        "strict_precision": sP,
        "strict_recall": sR,
        "strict_f1": sF1,
        "lenient_precision": lP,
        "lenient_recall": lR,
        "lenient_f1": lF1,
    }

Training

In [56]:
import random, os
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

batch_size = 8
grad_accum = 1  # effective batch = 16

training_args = TrainingArguments(
    seed=SEED,
    data_seed=SEED,
    output_dir=str(BASE / "distilbert_skill_ner_ckpts_v3"),
    num_train_epochs=8,              # let early stopping decide
    learning_rate=3e-5,               # changed
    warmup_ratio=0.00,                # NEW
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,  # NEW
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="strict_f1",
    greater_is_better=True,

    logging_steps=20,
    report_to=[],
    fp16=torch.cuda.is_available(),
    save_total_limit=2,               # NEW: keeps drive clean
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

train_out = trainer.train()
dev_metrics = trainer.evaluate(tokenized["validation"])
test_metrics = trainer.evaluate(tokenized["test"])
print("DEV:", dev_metrics)
print("TEST:", test_metrics)

/tmp/ipython-input-396838715.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Strict Precision,Strict Recall,Strict F1,Lenient Precision,Lenient Recall,Lenient F1
1,0.002100,0.626148,0.430168,0.478261,0.452941,0.705202,0.757764,0.730539
2,0.005400,0.664860,0.471429,0.409938,0.438538,0.793103,0.571429,0.664260
3,0.004200,0.619405,0.491228,0.521739,0.506024,0.721212,0.739130,0.730061
4,0.008000,0.575331,0.452514,0.503106,0.476471,0.730539,0.757764,0.743902
5,0.003900,0.578501,0.476471,0.503106,0.489426,0.726708,0.726708,0.726708


DEV: {'eval_loss': 0.6194054484367371, 'eval_strict_precision': 0.49122807017543857, 'eval_strict_recall': 0.5217391304347826, 'eval_strict_f1': 0.5060240963855421, 'eval_lenient_precision': 0.7212121212121212, 'eval_lenient_recall': 0.7391304347826086, 'eval_lenient_f1': 0.7300613496932515, 'eval_runtime': 0.3756, 'eval_samples_per_second': 279.536, 'eval_steps_per_second': 37.271, 'epoch': 5.0}
TEST: {'eval_loss': 0.457916259765625, 'eval_strict_precision': 0.5989847715736041, 'eval_strict_recall': 0.5645933014354066, 'eval_strict_f1': 0.58128078817734, 'eval_lenient_precision': 0.8222222222222222, 'eval_lenient_recall': 0.7081339712918661, 'eval_lenient_f1': 0.7609254498714654, 'eval_runtime': 0.4427, 'eval_samples_per_second': 237.159, 'eval_steps_per_second': 31.621, 'epoch': 5.0}


Save metrics + best model

In [57]:
BEST_DIR = BASE / "distilbert_runA_best"
trainer.save_model(str(BEST_DIR))
tokenizer.save_pretrained(str(BEST_DIR))

metrics = {"dev": dev_metrics, "test": test_metrics}
with open(BASE / "distilbert_runA_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=float)

Save token-level predictions for error analysis

In [58]:
pred = trainer.predict(tokenized["test"])
logits, labels = pred.predictions, pred.label_ids
pred_tags, gold_tags = align_predictions(logits, labels)

test_tokens = test_df["tokens"].tolist()

pred_df = pd.DataFrame({
    "tokens": test_tokens,
    "gold_tags": gold_tags,
    "pred_tags": pred_tags,
})

pred_df.to_csv(BASE / "distilbert_runA_test_wordlevel_preds.csv", index=False)
print("Saved:", BEST_DIR)

Saved: /content/drive/MyDrive/CS685/linkedin/distilbert_runA_best


FINAL RUN

In [59]:
full_train = concatenate_datasets([tokenized["train"], tokenized["validation"]])

model_final = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args_final = TrainingArguments(
    output_dir=str(BASE / "distilbert_final_ckpts"),
    num_train_epochs=8,
    learning_rate=3e-5,
    warmup_ratio=0.0,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    logging_steps=20,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

trainer_final = Trainer(
    model=model_final,
    args=training_args_final,
    train_dataset=full_train,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_final.train()
final_test_metrics = trainer_final.evaluate(tokenized["test"])
print("FINAL TEST:", final_test_metrics)

FINAL_DIR = BASE / "distilbert_skill_ner_final"
trainer_final.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

with open(BASE / "distilbert_skill_ner_final_metrics.json", "w") as f:
    json.dump({"test": final_test_metrics}, f, indent=2, default=float)

print("Saved final:", FINAL_DIR)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-294393111.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_final = Trainer(


Step,Training Loss
20,0.480200
40,0.277400
60,0.224700
80,0.239300
100,0.178100
120,0.185100
140,0.171600
160,0.121300
180,0.128700
200,0.100200


FINAL TEST: {'eval_loss': 0.30023691058158875, 'eval_strict_precision': 0.6201923076923077, 'eval_strict_recall': 0.6172248803827751, 'eval_strict_f1': 0.6187050359712231, 'eval_lenient_precision': 0.8256410256410256, 'eval_lenient_recall': 0.7703349282296651, 'eval_lenient_f1': 0.7970297029702972, 'eval_runtime': 0.3094, 'eval_samples_per_second': 339.421, 'eval_steps_per_second': 45.256, 'epoch': 8.0}
Saved final: /content/drive/MyDrive/CS685/linkedin/distilbert_skill_ner_final


Convert BIO tags to span

In [60]:
def bio_to_spans(tokens, tags, label="SKILL"):
    spans = []
    current = []

    for tok, tag in zip(tokens, tags):
        if tag == f"B-{label}":
            if current:
                spans.append(" ".join(current))
            current = [tok]
        elif tag == f"I-{label}":
            if current:
                current.append(tok)
            else:
                current = [tok]  # malformed I-, treat as B-
        else:  # O
            if current:
                spans.append(" ".join(current))
                current = []

    if current:
        spans.append(" ".join(current))

    # dedupe, preserve order
    seen = set()
    uniq = []
    for s in spans:
        if s not in seen:
            seen.add(s)
            uniq.append(s)

    return uniq

In [61]:
def predict_spans(model, tokenizer, tokens):
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
    )

    enc = {k: v.to(model.device) for k, v in enc.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**enc)

    pred_ids = outputs.logits.argmax(-1)[0].cpu().numpy()

    word_ids = tokenizer(tokens, is_split_into_words=True).word_ids()
    pred_tags = []
    seen = set()

    for i, w_id in enumerate(word_ids):
        if w_id is None or w_id in seen:
            continue
        seen.add(w_id)
        pred_tags.append(id2label[pred_ids[i]])

    return pred_tags

In [62]:
def predict_tags_wordlevel(model, tokenizer, tokens):
    model.eval()

    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
    )
    word_ids = enc.word_ids(batch_index=0)   # from same encoding

    enc = {k: v.to(model.device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)

    pred_ids = outputs.logits.argmax(-1)[0].detach().cpu().numpy()

    # take first subword per word
    pred_tags = []
    seen = set()
    for i, w_id in enumerate(word_ids):
        if w_id is None or w_id in seen:
            continue
        seen.add(w_id)
        pred_tags.append(id2label[int(pred_ids[i])])

    return pred_tags

In [90]:
examples = []

for i in [4, 16, 20, 30, 35, 80, 45, 65, 75, 103]:
    row = test_df.iloc[i]

    tokens = row["tokens"]
    gold_tags = row["tags"]

    pred_tags = predict_tags_wordlevel(trainer.model, tokenizer, tokens)

    examples.append({
        "sentence": " ".join(tokens),
        "gold_spans": bio_to_spans(tokens, gold_tags),
        "predicted_spans": bio_to_spans(tokens, pred_tags),
    })


examples_df = pd.DataFrame(examples)

# show full text
pd.set_option("display.max_colwidth", None)
display(examples_df)

# save for report
out_path = BASE / "runA_examples_full.csv"
examples_df.to_csv(out_path, index=False)
print("Saved:", out_path)

,sentence,gold_spans,predicted_spans
0,"The following skills / experience are not required , but would be considered a strong plus : Understanding of Sync / Async concepts ; Experience with uWSGI / ASGI , websocketsExperience with other languages such as C / C++ , Java , JavaScript , etc .","[Sync / Async concepts, uWSGI / ASGI, websocketsExperience, C / C++, Java, JavaScript]","[Sync / Async concepts, uWSGI / ASGI, websocketsExperience, C / C++, Java, JavaScript]"
1,"Good knowledge of Oracle 12C/19C server . Advance knowledge of Sun Solaris studio 12 , DBXtool , C++ compiler flags and configurations . Advance knowledge of PERL , SHELL scripting . Advance knowledge of Oops concept , design patterns , event driven program architecture .","[Oracle 12C/19C server, Sun Solaris studio 12, DBXtool, C++, compiler flags and configurations, PERL, SHELL scripting, Oops concept , design patterns , event driven program architecture]","[Oracle 12C/19C server, Sun Solaris studio 12, DBXtool, C++ compiler flags and configurations, PERL, SHELL scripting, Oops concept, design patterns]"
2,"We are looking for new colleagues who bring innovative ways of thinking and problem solving , and who want risks to be part of the team that changes the world ’s financial markets .",[],[]
3,"Preferred Qualifications Strong communication skills , both written and verbal , to effectively collaborate with team members and stakeholders .",[communication skills],"[communication skills, written and verbal]"
4,) Knowledge of integration technologies ( e.g.,[integration technologies],[integration technologies]
5,Familiarity with cloud computing platforms such as AWS or Google Cloud5 .,"[cloud computing platforms, AWS, Google Cloud5]","[cloud computing platforms, AWS, Google Cloud5]"
6,Experience with Postman collections for testing .,[Postman collections],[Postman]
7,6 Deployment Proficient in Vlcoity / Omnistudio deployment from one org to another .,[Vlcoity / Omnistudio deployment],"[Vlcoity, Omnistudio]"
8,"Key Qualifications : Demonstrable experience with large - scale cloud services deployment and management , along with a track record of maintained or improved system stability and scalability . Expert knowledge of Python web - app frameworks like Django and Flask .","[large - scale cloud services deployment and management, Python web - app frameworks, Django, Flask]","[large - scale cloud services deployment and management, Python web - app frameworks, Django, Flask]"
9,Passion and experience with intelligent video and audio processing is a must for this position .,[intelligent video and audio processing],[intelligent video and audio processing]


Saved: /content/drive/MyDrive/CS685/linkedin/runA_examples_full.csv
